# Hiệu chuẩn LLM-as-judge với nhãn người

Module 13 mục 6 kiểm judge bằng nguồn sai trên 13 câu và khuyên "hiệu chỉnh trên 30 tới 50 câu người chấm". Bài này làm
việc đó cho đúng, trên nhãn người tái lập được:

- **Pointwise**: 150 câu UIT-VSFC (tập ghim của module 11), mỗi câu kèm một nhãn cảm xúc "hệ thống gán" mà một nửa bị
  đổi sai có chủ đích. Judge trả lời nhãn đó đúng không. Nhãn người (gold của UIT-VSFC) cho biết đáp án thật, nên đo được
  accuracy, Cohen's kappa, judge sai theo hướng nào, và xác suất judge đưa ra có đúng nghĩa xác suất không.
- **Pairwise**: 60 câu hỏi Zalo Legal, judge chọn đoạn nào trả lời tốt hơn giữa điều luật đúng (qrels) và một điều luật
  gần nghĩa nhưng sai (hard negative từ dense retrieval). Đổi chỗ hai đoạn để đo thiên vị vị trí, độn chữ để đo thiên vị
  độ dài.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b` và `bge-m3`, bộ `uit-vsfc` và `zalo-legal` (cache `bge-m3.npy` của module 13 bài
  Intermediate 01). Khoảng 600 lượt gọi `qwen2.5:3b`, khoảng 5 phút trên PC của track (GPU dùng chung).
  Judge `qwen2.5:7b` (tuỳ chọn) chạy trên CPU khoảng 7 phút trên PC của track.
- Chấm: `uv run pytest tracks/ai-engineer/modules/20-evaluation-production/exercises/intermediate/01-judge-calibration -q`

In [ ]:
# explore: đưa gốc repo vào sys.path để import llmkit, tìm thư mục dữ liệu
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
VSFC = ROOT / "data" / "raw" / "uit-vsfc"
ZALO = ROOT / "data" / "raw" / "zalo-legal"

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import math

import numpy as np

## 1. Dán lời giải bài Basic 02

Notebook tự chứa: dán `VERDICTS`, `cohen_kappa`, `calibration_report`, `resolve_swap`, `pairwise_summary` của bài Basic 02
vào cell dưới (giữ `from collections import Counter`). Test kiểm lại vài trường hợp.

In [ ]:
from collections import Counter

VERDICTS = ("first", "second", "tie")

# TODO: dán cohen_kappa, calibration_report, resolve_swap, pairwise_summary của bài Basic 02 vào đây.

## 2. Judge pointwise: nhãn hệ thống gán có đúng không

`make_items(gold, seed=0, flip_rate=0.5) -> (candidates, truth)`: với `rng = np.random.default_rng(seed)`, duyệt `gold`
theo thứ tự; mỗi câu gọi `rng.random()` một lần: nhỏ hơn `flip_rate` thì nhãn ứng viên là một nhãn **khác** gold, chọn
bằng `rng.integers(0, 2)` trong hai nhãn còn lại theo thứ tự của `LABELS`, và `truth` là `False`; còn lại nhãn ứng viên
là gold, `truth` là `True`. Đúng thứ tự gọi RNG này thì tập của bạn trùng tập của lời giải.

`judge_messages(sentence, candidate) -> list[dict]`: hai message, `system` = `JUDGE_SYSTEM`, `user` =
`"Phản hồi: <sentence>\nNhãn hệ thống gán: <candidate>"`.

`parse_yes_no(reply) -> bool | None`: bỏ khoảng trắng hai đầu, bỏ dấu chấm cuối, chữ thường; `"yes"` cho `True`, `"no"`
cho `False`, còn lại `None`.

`p_yes(top_logprobs) -> float | None`: `top_logprobs` là list dict `{"token", "logprob"}` của **token output đầu tiên**
(Ollama trả khi gửi `"logprobs": true, "top_logprobs": 5`). Cộng $$e^{\text{logprob}}$$ của các token mà `strip().lower()`
là `"yes"`, tương tự cho `"no"`; trả `yes / (yes + no)`. Không có token nào như vậy (hoặc list rỗng, `None`) thì `None`.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
JUDGE_SYSTEM = (
    "Bạn là người kiểm tra nhãn. Một hệ thống đã gán nhãn cảm xúc cho câu phản hồi của sinh viên dưới đây theo định nghĩa:\n"
    "- negative: chê, phàn nàn, hoặc góp ý chỉ ra điều chưa tốt.\n"
    "- neutral: không khen không chê, chỉ mô tả, hỏi, cảm ơn chung chung, hoặc không có ý kiến.\n"
    "- positive: khen, hài lòng.\n"
    "Hỏi: nhãn hệ thống gán có đúng không? Chỉ trả lời đúng một từ: yes hoặc no."
)


def make_items(gold: list[str], seed: int = 0, flip_rate: float = 0.5) -> tuple[list[str], list[bool]]:
    """Nhãn ứng viên (một phần bị đổi sai có chủ đích) và truth = ứng viên có trùng gold không."""
    # TODO
    raise NotImplementedError


def judge_messages(sentence: str, candidate: str) -> list[dict]:
    # TODO
    raise NotImplementedError


def parse_yes_no(reply: str) -> bool | None:
    # TODO
    raise NotImplementedError


def p_yes(top_logprobs) -> float | None:
    """P(yes) chuẩn hoá trên hai token yes/no trong top logprobs của token đầu."""
    # TODO
    raise NotImplementedError

## 3. Xác suất của judge có đáng tin không

Judge nói "yes" với xác suất 0.99 có đúng 99% không? `expected_calibration_error(probs, truth, n_bins=10) -> float`: bỏ
cặp có `probs` là `None`; chia [0, 1] thành `n_bins` khoảng đều $$[b/n, (b+1)/n)$$, khoảng cuối gồm cả 1.0; với mỗi khoảng
không rỗng $$B$$, cộng $$\frac{|B|}{N}\,\left|\overline{p}_B - \overline{y}_B\right|$$ (xác suất trung bình trừ tỉ lệ `True`
thật trong khoảng). `ValueError` khi không còn cặp nào.

`pick_threshold(scores, truth) -> float`: chọn ngưỡng $$c$$ để quyết định "qua" khi `score >= c`, tối đa hoá accuracy trên
tập dev. Ứng viên là mọi giá trị khác nhau của `scores` (bỏ `None`) theo thứ tự tăng dần, cộng thêm một giá trị lớn hơn
mọi điểm (`max + 1e-9`, nghĩa là "không cho qua câu nào"); bằng nhau thì lấy ứng viên **nhỏ nhất**. `ValueError` khi không có
điểm nào. Dùng nó đúng cách: chọn ngưỡng trên nửa đầu, báo accuracy trên nửa sau.

In [ ]:
def expected_calibration_error(probs, truth, n_bins: int = 10) -> float:
    # TODO
    raise NotImplementedError


def pick_threshold(scores, truth) -> float:
    # TODO
    raise NotImplementedError

## 4. Judge pairwise có đổi chỗ

`pair_messages(question, first, second) -> list[dict]`: `system` = `PAIR_SYSTEM`, `user` =
`"Câu hỏi: <question>\n\nĐoạn 1:\n<first>\n\nĐoạn 2:\n<second>"`.

`parse_choice(reply) -> str | None`: `json.loads` cả chuỗi (sau `strip`), phải là object có `choice` thuộc `"1"`, `"2"`,
`"tie"`; đổi thành `"first"`, `"second"`, `"tie"`. Mọi trường hợp khác (không phải JSON, thiếu khoá, giá trị lạ, kiểu lạ)
trả `None`.

`judge_pair(chat, question, a, b) -> (v_ab, v_ba)`: `chat(messages) -> str` là hàm gọi judge (đã gắn `PAIR_SCHEMA`).
Gọi đúng hai lần, lần đầu `a` là Đoạn 1, lần hai `b` là Đoạn 1; trả hai verdict đã `parse_choice`.

In [ ]:
PAIR_SYSTEM = (
    "Bạn đánh giá văn bản pháp luật. Cho một câu hỏi và hai đoạn văn bản (Đoạn 1, Đoạn 2). Đoạn nào chứa căn cứ "
    "pháp lý trả lời câu hỏi tốt hơn? Trả lời bằng JSON với khoá `choice` là \"1\", \"2\" hoặc \"tie\" nếu ngang nhau."
)
PAIR_SCHEMA = {"type": "object", "properties": {"choice": {"type": "string", "enum": ["1", "2", "tie"]}},
               "required": ["choice"], "additionalProperties": False}


def pair_messages(question: str, first: str, second: str) -> list[dict]:
    # TODO
    raise NotImplementedError


def parse_choice(reply: str) -> str | None:
    # TODO
    raise NotImplementedError


def judge_pair(chat, question: str, a: str, b: str) -> tuple[str | None, str | None]:
    # TODO
    raise NotImplementedError

## 5. Chạy thật

Client nhỏ dưới đây gọi thẳng `/api/chat` của Ollama (`temperature` 0, `seed` 0) để lấy `logprobs`; `num_predict` 2 vì
judge chỉ cần một từ. Tập ghim UIT-VSFC dựng y như module 11 bài Intermediate 02.

In [ ]:
# explore: client Ollama có logprobs, tập ghim 150 câu, nhãn ứng viên
import time

import httpx
import pandas as pd

OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
http = httpx.Client(timeout=900)


def ollama_chat(messages, model="qwen2.5:3b", cpu=False, num_predict=2, logprobs=True, fmt=None):
    options = {"temperature": 0, "seed": 0, "num_predict": num_predict}
    if cpu:
        options.update(num_gpu=0, num_thread=4)
    payload = {"model": model, "messages": messages, "stream": False, "options": options}
    if logprobs:
        payload.update(logprobs=True, top_logprobs=5)
    if fmt is not None:
        payload["format"] = fmt
    r = http.post(OLLAMA + "/api/chat", json=payload)
    r.raise_for_status()
    return r.json()


test = pd.read_parquet(VSFC / "default" / "test" / "0000.parquet")
parts = [test[test.sentiment == c].sample(50, random_state=0) for c in range(3)]
pinned = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
sentences = list(pinned.sentence)
gold = [LABELS[s] for s in pinned.sentiment]
candidates, truth = make_items(gold, seed=0)
print(len(sentences), "câu; nhãn ứng viên đúng:", sum(truth))

In [ ]:
# explore: judge pointwise qwen2.5:3b (GPU nếu có), khoảng 1 phút
def run_pointwise(model, cpu=False):
    rows = []
    for s, c, t, g in zip(sentences, candidates, truth, gold):
        start = time.perf_counter()
        data = ollama_chat(judge_messages(s, c), model=model, cpu=cpu)
        top = data["logprobs"][0]["top_logprobs"] if data.get("logprobs") else None
        rows.append(dict(sentence=s, gold=g, cand=c, truth=t, reply=data["message"]["content"],
                         verdict=parse_yes_no(data["message"]["content"]), p_yes=p_yes(top),
                         seconds=time.perf_counter() - start))
    return pd.DataFrame(rows)


def report(df, name):
    rep = calibration_report(list(df.verdict), list(df.truth))
    half = len(df) // 2
    thr = pick_threshold(list(df.p_yes[:half]), list(df.truth[:half]))
    test_rows = df[half:][df.p_yes[half:].notna()]
    rep.update(ece=expected_calibration_error(list(df.p_yes), list(df.truth)),
               thr_dev=thr, acc_test_thr=float(((test_rows.p_yes >= thr) == test_rows.truth).mean()),
               acc_test_05=float(((test_rows.p_yes >= 0.5) == test_rows.truth).mean()),
               p50_s=float(df.seconds.median()))
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in rep.items()})
    return rep


point3 = run_pointwise("qwen2.5:3b")
rep3 = report(point3, "qwen2.5:3b")

In [ ]:
# explore: judge sai ở đâu? false pass theo nhãn ứng viên, false fail theo nhãn gold
def where_wrong(df):
    fp = df[~df.truth].groupby("cand").verdict.apply(lambda v: f"{int((v == True).sum())}/{len(v)}")
    ff = df[df.truth].groupby("gold").verdict.apply(lambda v: f"{int((v == False).sum())}/{len(v)}")
    return pd.DataFrame({"false_pass (theo nhãn ứng viên)": fp, "false_fail (theo nhãn gold)": ff})


print(where_wrong(point3))
for r in point3[(point3.truth) & (point3.verdict == False)].head(5).itertuples():
    print(f"  bị từ chối dù đúng: [{r.gold}] {r.sentence}")

In [ ]:
# explore: biểu đồ độ tin cậy (reliability) của P(yes)
import matplotlib.pyplot as plt

def reliability(df, ax, name, n_bins=5):
    d = df[df.p_yes.notna()]
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(d.p_yes, edges) - 1, 0, n_bins - 1)
    xs = [d.p_yes[idx == b].mean() for b in range(n_bins) if (idx == b).any()]
    ys = [d.truth[idx == b].mean() for b in range(n_bins) if (idx == b).any()]
    ns = [int((idx == b).sum()) for b in range(n_bins) if (idx == b).any()]
    ax.plot([0, 1], [0, 1], color="#999", lw=1)
    ax.scatter(xs, ys, s=[10 + 2 * n for n in ns])
    ax.set(title=name, xlabel="P(yes) trung bình", ylabel="tỉ lệ đúng thật", xlim=(0, 1), ylim=(0, 1))


fig, ax = plt.subplots(figsize=(4, 4))
reliability(point3, ax, "qwen2.5:3b")
plt.show()

In [ ]:
# explore: (tuỳ chọn) judge qwen2.5:7b chạy CPU 4 luồng (num_gpu 0), khoảng 7 phút trên PC của track.
# Trên laptop GPU 4GB, bỏ cpu=True để Ollama tự chia GPU/CPU. Đặt M20_SKIP_7B=1 để bỏ qua.
point7 = None
if os.environ.get("M20_SKIP_7B") != "1":
    point7 = run_pointwise("qwen2.5:7b", cpu=True)
    rep7 = report(point7, "qwen2.5:7b")
    print(where_wrong(point7))
    both = pd.DataFrame({"3b": point3.verdict, "7b": point7.verdict, "truth": point3.truth})
    print("kappa giữa hai judge:", round(cohen_kappa(list(both["3b"]), list(both["7b"])), 3))

In [ ]:
# explore: cặp Zalo: điều luật đúng (qrels) và hard negative (dense top-20 đầu tiên không thuộc qrels)
import llmkit

qrels, order = {}, []
with (ZALO / "qrels" / "test.jsonl").open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["query-id"] not in qrels:
            order.append(d["query-id"])
        qrels.setdefault(d["query-id"], []).append(d["corpus-id"])
with (ZALO / "queries.jsonl").open(encoding="utf-8") as f:
    qtext = {d["_id"]: d["text"] for d in map(json.loads, f)}
with (ZALO / "corpus.jsonl").open(encoding="utf-8") as f:
    corpus = {d["_id"]: d["title"] + "\n" + d["text"] for d in map(json.loads, f)}
ids = json.loads((ZALO / "cache" / "bge-m3.ids.json").read_text())
emb = np.load(ZALO / "cache" / "bge-m3.npy").astype(np.float32)
emb /= np.linalg.norm(emb, axis=1, keepdims=True)

qids = order[:60]
qv = llmkit.embed([qtext[q] for q in qids])
qv /= np.linalg.norm(qv, axis=1, keepdims=True)
items = []
for q, v in zip(qids, qv):
    top = np.argsort(-(emb @ v))[:20]
    neg = next(ids[i] for i in top if ids[i] not in qrels[q])
    items.append(dict(question=qtext[q], gold=corpus[qrels[q][0]][:1000], neg=corpus[neg][:1000]))
print(len(items), "cặp; ví dụ:", items[0]["question"])

In [ ]:
# explore: ba điều kiện x hai thứ tự x 60 cặp = 360 lượt gọi qwen2.5:3b, khoảng 3 phút
PAD = ("\nViệc áp dụng quy định này phải bảo đảm tuân thủ đúng trình tự, thủ tục theo quy định của pháp luật hiện hành; "
       "cơ quan, tổ chức, cá nhân có liên quan có trách nhiệm phối hợp thực hiện nghiêm túc, đầy đủ, kịp thời và hiệu quả, "
       "bảo đảm quyền và lợi ích hợp pháp của các bên theo đúng quy định.") * 2


def pair_chat(messages):
    return ollama_chat(messages, num_predict=16, logprobs=False, fmt=PAIR_SCHEMA)["message"]["content"]


conditions = {
    "gốc": lambda it: (it["gold"], it["neg"]),
    "độn đoạn sai": lambda it: (it["gold"], it["neg"] + PAD),
    "độn đoạn đúng": lambda it: (it["gold"] + PAD, it["neg"]),
}
pair_rows = []
for name, make in conditions.items():
    for i, it in enumerate(items):
        a, b = make(it)
        v_ab, v_ba = judge_pair(pair_chat, it["question"], a, b)
        pair_rows.append(dict(condition=name, i=i, v_ab=v_ab, v_ba=v_ba))
pairs = pd.DataFrame(pair_rows)
print("verdict không đọc được:", int(pairs.v_ab.isna().sum() + pairs.v_ba.isna().sum()))
summary = {name: pairwise_summary([(r.v_ab, r.v_ba) for r in g.itertuples() if r.v_ab and r.v_ba])
           for name, g in pairs.groupby("condition", sort=False)}
print(pd.DataFrame(summary).T.round(3))

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Accuracy và kappa của judge 3B là bao nhiêu? Judge sai theo hướng nào nhiều hơn: cho qua nhãn sai (false pass) hay từ
  chối nhãn đúng (false fail)? Lớp nào chịu nhiều nhất, và điều đó khớp với lớp khó nhất ở module 11 mục 7 không?
- Biểu đồ độ tin cậy: judge 3B tự tin quá hay thiếu tự tin? Ngưỡng chọn trên nửa đầu khác 0.5 bao nhiêu, và trên nửa
  sau nó có thắng ngưỡng 0.5 không? Một chênh lệch vài câu trên 75 câu có nói được gì không?
- (Nếu chạy 7B) Judge lớn hơn đổi kappa, false pass và ECE thế nào? Hai judge khớp nhau tới đâu?
- Pairwise: judge chọn đúng điều luật bao nhiêu phần trăm, nhất quán khi đổi chỗ bao nhiêu phần trăm, thích vị trí nào?
- Độn chữ vô nghĩa vào đoạn sai hay đoạn đúng làm tỉ lệ thắng của đoạn đúng đổi thế nào? Đó có phải thiên vị độ dài theo
  nghĩa của Zheng et al. (2023) không, hay là thứ khác?

Sau đó làm phần trắc nghiệm của bài trong app.